# 01 - MediaPipe Keypoint Extraction (Arabic)
This notebook extracts 63-dimensional keypoints (21 landmarks x 3 coords) from hand gesture images using MediaPipe Hands.


In [2]:
import os
import cv2
import glob
import json
import numpy as np
import pandas as pd
import mediapipe as mp


## Configuration


In [3]:
# Detect environment
IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    print("Environment: Kaggle detected")
    DATASET_DIR = '/kaggle/input/arabic-sign-language-letters' # Update this path based on uploaded Kaggle dataset
else:
    print("Environment: Local machine detected")
    DATASET_DIR = './dataset' # Local dataset path

OUTPUT_CSV = 'arabic_mediapipe_keypoints.csv'
INCLUDE_ZERO_ROW_IF_NO_HAND = True
MAX_IMAGES_PER_CLASS = None # set to integer like 1000 to limit
REPORT_JSON = 'extraction_report.json'


Environment: Local machine detected


## Initialize MediaPipe


In [4]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)


## Process Images and Extract Keypoints


In [5]:
cols = ['label']
for i in range(21):
    cols.extend([f'x{i}', f'y{i}', f'z{i}'])

csv_data = []
stats = {
    'total_images': 0,
    'successful': 0,
    'no_hand': 0,
    'per_class': {}
}

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f"DATASET_DIR '{DATASET_DIR}' not found. Please verify the path.")

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f"Found {len(classes)} classes.")

for cls in classes:
    cls_dir = os.path.join(DATASET_DIR, cls)
    images = glob.glob(os.path.join(cls_dir, '*.jpg')) + glob.glob(os.path.join(cls_dir, '*.png'))
    if MAX_IMAGES_PER_CLASS:
        images = images[:MAX_IMAGES_PER_CLASS]
        
    stats['per_class'][cls] = {'total': len(images), 'success': 0}
    stats['total_images'] += len(images)
    
    for img_path in images:
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)
        
        if results.multi_hand_landmarks:
            landmarks = results.multi_hand_landmarks[0]
            row = [cls]
            for lm in landmarks.landmark:
                row.extend([lm.x, lm.y, lm.z])
            csv_data.append(row)
            stats['successful'] += 1
            stats['per_class'][cls]['success'] += 1
        else:
            stats['no_hand'] += 1
            if INCLUDE_ZERO_ROW_IF_NO_HAND:
                row = [cls] + [0.0] * 63
                csv_data.append(row)

hands.close()

# Save to CSV
df = pd.DataFrame(csv_data, columns=cols)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")

# Save Report
with open(REPORT_JSON, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=4, ensure_ascii=False)

print("Extraction Statistics:")
print(json.dumps(stats, indent=4, ensure_ascii=False))


FileNotFoundError: DATASET_DIR './dataset' not found. Please verify the path.